In [ ]:
# Step 6.1: imports and PostgreSQL connection
import os
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

load_dotenv()

DB_USER = os.getenv("POSTGRES_USER", "postgres")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB")




if not DB_PASSWORD or not DB_NAME:
    raise RuntimeError(
        "Set POSTGRES_PASSWORD and POSTGRES_DB before running this notebook."
    )

engine = create_engine(
    "postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as connection:
    print(connection.execute(text("SELECT 1")).scalar())
 # Step 6.2: load analytical views
view_queries = {
    "customer_360": "SELECT * FROM vw_customer_360",
    "customer_rfm": "SELECT * FROM vw_rfm_segments",
    "monthly_revenue": "SELECT * FROM vw_monthly_revenue",
    "cohort_retention": "SELECT * FROM vw_cohort_retention",
    "customer_churn": "SELECT * FROM vw_customer_churn",
    "customer_risk": "SELECT * FROM vw_customer_risk",
}

dataframes = {
    name: pd.read_sql_query(query, engine)
    for name, query in view_queries.items()
}

customer_360 = dataframes["customer_360"]
customer_rfm = dataframes["customer_rfm"]
monthly_revenue = dataframes["monthly_revenue"]
cohort_retention = dataframes["cohort_retention"]
customer_churn = dataframes["customer_churn"]
customer_risk = dataframes["customer_risk"]

for name, dataframe in dataframes.items():
    print(f"{name}: {dataframe.shape}")
 # Step 6.3: inspect structure and missing values
customer_360.info()
customer_360.head()
customer_360.describe(include="all")

missing_values = customer_360.isna().sum().sort_values(ascending=False).to_frame(
    "missing_count"
)
missing_values["missing_percentage"] = (
    missing_values["missing_count"] / len(customer_360) * 100
)
missing_values
 # Step 6.4: validation checks
validation_results = {
    "duplicate_customer_ids": int(
        customer_360["customer_unique_id"].duplicated().sum()
    ),
    "negative_revenue": int((customer_360["total_revenue"] < 0).sum()),
    "negative_orders": int((customer_360["total_orders"] < 0).sum()),
    "negative_aov": int((customer_360["average_order_value"] < 0).sum()),
}

for check, count in validation_results.items():
    print(f"{check}: {count}")

for column in ["recency_score", "frequency_score", "monetary_score"]:
    print(column, sorted(customer_rfm[column].dropna().unique()))

assert validation_results["duplicate_customer_ids"] == 0
assert validation_results["negative_revenue"] == 0
assert validation_results["negative_orders"] == 0
assert validation_results["negative_aov"] == 0

print("Validation checks passed.")

1
customer_360: (96096, 9)
customer_rfm: (93358, 8)
monthly_revenue: (23, 5)
cohort_retention: (219, 5)
customer_churn: (93358, 5)
customer_risk: (93358, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_unique_id   96096 non-null  object        
 1   customer_city        96096 non-null  object        
 2   customer_state       96096 non-null  object        
 3   total_orders         96096 non-null  int64         
 4   delivered_orders     96096 non-null  int64         
 5   total_revenue        96096 non-null  float64       
 6   average_order_value  93358 non-null  float64       
 7   first_order_date     96096 non-null  datetime64[ns]
 8   last_order_date      96096 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(2), object(3)
memory usage: 6.6+ MB
duplicate_customer_ids: 0
n